## Option C — Ensemble, TTA, and GPU-optimized training (03b)

This notebook extends [03-deep-learning-model-optionC.ipynb](03-deep-learning-model-optionC.ipynb) with:

- **Preprocessing checks** (train tensors in [0, 1]; ImageNet normalization on val and after train aug).
- **Multiple backbones** (ResNet-50, EfficientNet-B0, optional ConvNeXt-Tiny), each trained with the **same progressive schedule** as notebook 03.
- **AMP**, **cuDNN benchmark**, optional **torch.compile**, **DataLoader workers** + **pin_memory**, and **batched** torchvision **v2** augmentations on GPU.
- **Ensemble** inference: average **dog-class** probabilities across models; **TTA** (identity + horizontal flip); **threshold** tuning on the fixed val split.

**Leaderboard note:** Higher accuracy is the goal; **100%** on a hidden test set is **not** guaranteed.

**Runtime:** Progressive training × several architectures is **long**. Set `QUICK_DEBUG = True` to use only the full training-set size once per architecture.

**GPU:** During training, use `nvidia-smi` (or Activity Monitor on Mac) to confirm the GPU stays busy.


In [ ]:
from pathlib import Path
import os
import sys

# Project root = parent of notebooks/
sys.path.insert(0, str(Path().resolve().parent))

from src.config import PART1_TRAIN_DIR, PART1_TEST_DIR

assert PART1_TRAIN_DIR.exists(), f"Training data missing: {PART1_TRAIN_DIR}"
assert PART1_TEST_DIR.exists(), f"Test data missing: {PART1_TEST_DIR}"

print("Data paths OK")
print(f"  Train: {PART1_TRAIN_DIR}")
print(f"  Test:  {PART1_TEST_DIR}")


### Imports, device, and constants


In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision.transforms import v2 as Tv2
from tqdm import tqdm
from sklearn.utils import resample

from src.config import IMG_SIZE_CNN, MODELS_DIR, OUTPUTS_DIR
from src.utils import (
    load_labeled_images,
    split_data,
    get_pytorch_dataloaders,
    load_test_images,
    generate_submission_csv,
)

# --- Runtime / training knobs -------------------------------------------------
QUICK_DEBUG = False
# Full progressive schedule (same idea as 03). If QUICK_DEBUG, only the last stage.
SAMPLE_SIZES_FULL = [500, 1000, 2000, 4000]

NUM_EPOCHS_HEAD = 10
NUM_EPOCHS_FULL = 14
BATCH_SIZE = 64

# Optional second ResNet-50 trained with a different seed (set to None to skip)
EXTRA_RESNET_SEED = None

# Threshold search on mean P(dog) across models (val)
THRESHOLD_GRID = np.arange(0.45, 0.56, 0.01)

# DataLoader parallelism (CPU). Set workers to 0 on Windows if you see spawn issues.
NUM_WORKERS = min(8, max(0, (os.cpu_count() or 4) - 1))
PIN_MEMORY = torch.cuda.is_available()
USE_AMP = torch.cuda.is_available()
# torch.compile can complicate checkpoint keys; leave False unless you tune save/load.
USE_TORCH_COMPILE = False

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

print(f"Device: {device}")
print(f"NUM_WORKERS={NUM_WORKERS}, PIN_MEMORY={PIN_MEMORY}, USE_AMP={USE_AMP}, USE_TORCH_COMPILE={USE_TORCH_COMPILE}")
print(f"BATCH_SIZE={BATCH_SIZE}, epochs head/full={NUM_EPOCHS_HEAD}/{NUM_EPOCHS_FULL}")
print("Monitor GPU load with nvidia-smi while training runs.")


### Load data and Step 1 — preprocessing checks


In [ ]:
X, y = load_labeled_images(img_size=IMG_SIZE_CNN, grayscale=False, max_samples=None)
print(f"Full dataset: {X.shape}, labels {y.shape}")

assert X.dtype == np.float32
assert float(X.min()) >= 0.0 and float(X.max()) <= 1.0, "Expected float32 images in [0, 1] from load_labeled_images."

X_train_full, X_val, y_train_full, y_val = split_data(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train_full.shape}, Val: {X_val.shape}")

if QUICK_DEBUG:
    SAMPLE_SIZES = [len(X_train_full)]
else:
    SAMPLE_SIZES = SAMPLE_SIZES_FULL + [len(X_train_full)]

print(f"Progressive sizes: {SAMPLE_SIZES}")

SUBMISSIONS_DIR = OUTPUTS_DIR / "submissions"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)


### Helper functions — models, batched aug (v2), train/eval

Training uses **normalize_train=False** in the loader so tensors stay in **[0, 1]**; **Tv2** augment + **Normalize** run on the **GPU** on the full batch. Validation tensors are normalized in the loader (ImageNet stats), matching pretrained backbones.


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def build_train_augment_v2(img_size: tuple[int, int]):
    # Batched GPU augment; NCHW [0,1] in, ImageNet-normalized out.
    h, w = img_size
    return Tv2.Compose(
        [
            Tv2.RandomHorizontalFlip(),
            Tv2.RandomRotation(15),
            Tv2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            Tv2.RandomResizedCrop((h, w), scale=(0.8, 1.0)),
            Tv2.RandomErasing(p=0.1),
            Tv2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )


def build_resnet50(dev: torch.device):
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    for p in m.parameters():
        p.requires_grad = False
    nf = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(nf, 2))
    return m.to(dev)


def build_efficientnet_b0(dev: torch.device):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    for p in m.parameters():
        p.requires_grad = False
    nf = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(nf, 2))
    return m.to(dev)


def build_convnext_tiny(dev: torch.device):
    m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    for p in m.parameters():
        p.requires_grad = False
    in_f = None
    for mod in m.head.modules():
        if isinstance(mod, nn.Linear):
            in_f = mod.in_features
    m.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, 2))
    return m.to(dev)


def build_ensemble_specs():
    out = [
        ("resnet50", build_resnet50, 42),
        ("efficientnet_b0", build_efficientnet_b0, 42),
    ]
    try:
        _ = models.ConvNeXt_Tiny_Weights.DEFAULT
        out.append(("convnext_tiny", build_convnext_tiny, 42))
    except Exception as e:
        print(f"Skipping ConvNeXt-Tiny: {e}")
    if EXTRA_RESNET_SEED is not None:
        out.append((f"resnet50_seed{EXTRA_RESNET_SEED}", build_resnet50, int(EXTRA_RESNET_SEED)))
    return out


def make_model_for_name(name: str, dev: torch.device):
    if name.startswith("resnet50"):
        return build_resnet50(dev)
    if name.startswith("efficientnet"):
        return build_efficientnet_b0(dev)
    if name.startswith("convnext"):
        return build_convnext_tiny(dev)
    return build_resnet50(dev)


def train_one_epoch(model, loader, criterion, optimizer, dev, train_aug, scaler, use_amp):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    cuda_amp = use_amp and dev.type == "cuda"

    for images, labels in tqdm(loader, desc="  Train", leave=False):
        images = images.to(dev, non_blocking=True)
        labels = labels.to(dev, non_blocking=True)
        images = train_aug(images)
        optimizer.zero_grad(set_to_none=True)
        if cuda_amp and scaler is not None:
            with torch.amp.autocast("cuda", enabled=True):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, dev, use_amp):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    cuda_amp = use_amp and dev.type == "cuda"

    for images, labels in tqdm(loader, desc="  Val  ", leave=False):
        images = images.to(dev, non_blocking=True)
        labels = labels.to(dev, non_blocking=True)
        if cuda_amp:
            with torch.amp.autocast("cuda", enabled=True):
                outputs = model(images)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


print("Model factories and train/eval helpers ready.")


### Progressive training (per architecture)


In [ ]:
ENSEMBLE_SPECS = build_ensemble_specs()
train_aug_gpu = build_train_augment_v2(IMG_SIZE_CNN)

saved_checkpoints = {}
all_arch_progressive = {}

for arch_name, build_fn, data_seed in ENSEMBLE_SPECS:
    print(f"\n{'#' * 70}\n# Architecture: {arch_name} (resample seed={data_seed})\n{'#' * 70}")
    torch.manual_seed(data_seed)
    np.random.seed(data_seed)

    progressive_results = []
    best_state_last_run = None
    t0 = time.time()

    for n_samples in SAMPLE_SIZES:
        print(f"\n{'=' * 60}\nTraining with {n_samples} images (of {len(X_train_full)})\n{'=' * 60}")

        if n_samples < len(X_train_full):
            X_sub, y_sub = resample(
                X_train_full,
                y_train_full,
                n_samples=n_samples,
                stratify=y_train_full,
                random_state=data_seed,
            )
        else:
            X_sub, y_sub = X_train_full, y_train_full

        train_loader, val_loader = get_pytorch_dataloaders(
            X_sub,
            y_sub,
            X_val,
            y_val,
            batch_size=BATCH_SIZE,
            img_size=IMG_SIZE_CNN,
            normalize_train=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            persistent_workers=NUM_WORKERS > 0,
            prefetch_factor=2 if NUM_WORKERS > 0 else None,
        )
        print(f"  batches train/val: {len(train_loader)} / {len(val_loader)}")

        model = build_fn(device)
        if USE_TORCH_COMPILE and device.type == "cuda":
            model = torch.compile(model)
        scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))
        criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

        head_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = optim.Adam(head_params, lr=1e-3, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS_HEAD)

        history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
        best_val_acc = 0.0
        best_state_run = None

        print(f"  Phase 1: head only, {NUM_EPOCHS_HEAD} epochs")
        for epoch in range(1, NUM_EPOCHS_HEAD + 1):
            tl, ta = train_one_epoch(
                model, train_loader, criterion, optimizer, device, train_aug_gpu, scaler, USE_AMP
            )
            vl, va = evaluate(model, val_loader, criterion, device, USE_AMP)
            scheduler.step()
            history["train_loss"].append(tl)
            history["val_loss"].append(vl)
            history["train_acc"].append(ta)
            history["val_acc"].append(va)
            print(f"    Ep {epoch}/{NUM_EPOCHS_HEAD} TrL:{tl:.4f} TrA:{ta:.4f} VL:{vl:.4f} VA:{va:.4f}")
            if va > best_val_acc:
                best_val_acc = va
                _sd = model._orig_mod.state_dict() if hasattr(model, "_orig_mod") else model.state_dict()
                best_state_run = {k: v.detach().cpu().clone() for k, v in _sd.items()}

        for param in model.parameters():
            param.requires_grad = True
        optimizer_ft = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
        scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=NUM_EPOCHS_FULL)

        print(f"  Phase 2: full fine-tune, {NUM_EPOCHS_FULL} epochs")
        for epoch in range(1, NUM_EPOCHS_FULL + 1):
            tl, ta = train_one_epoch(
                model, train_loader, criterion, optimizer_ft, device, train_aug_gpu, scaler, USE_AMP
            )
            vl, va = evaluate(model, val_loader, criterion, device, USE_AMP)
            scheduler_ft.step()
            history["train_loss"].append(tl)
            history["val_loss"].append(vl)
            history["train_acc"].append(ta)
            history["val_acc"].append(va)
            print(f"    Ep {epoch}/{NUM_EPOCHS_FULL} TrL:{tl:.4f} TrA:{ta:.4f} VL:{vl:.4f} VA:{va:.4f}")
            if va > best_val_acc:
                best_val_acc = va
                _sd = model._orig_mod.state_dict() if hasattr(model, "_orig_mod") else model.state_dict()
                best_state_run = {k: v.detach().cpu().clone() for k, v in _sd.items()}

        progressive_results.append({"n_train": n_samples, "best_val_acc": best_val_acc, "history": history})
        print(f"  best val acc (this stage): {best_val_acc:.4f}")
        best_state_last_run = best_state_run

    if hasattr(model, "_orig_mod"):
        model._orig_mod.load_state_dict(best_state_last_run)
    else:
        model.load_state_dict(best_state_last_run)
    ckpt_path = MODELS_DIR / f"optionC03b_{arch_name}_final.pth"
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(best_state_last_run, ckpt_path)
    saved_checkpoints[arch_name] = ckpt_path
    all_arch_progressive[arch_name] = progressive_results

    print(f"\nSaved {ckpt_path}")
    print(f"  wall time (this architecture): {(time.time() - t0) / 60.0:.2f} min")
    print("  Progressive summary:")
    for r in progressive_results:
        print(f"    {r['n_train']:>6d} images -> val acc {r['best_val_acc']:.4f}")

print("\nAll architectures trained. Checkpoints:", list(saved_checkpoints.keys()))


### Optional — val misclassifications (first checkpoint)


In [ ]:
# Re-load first saved architecture and list a few wrong val indices.

if saved_checkpoints:
    first_name = next(iter(saved_checkpoints.keys()))
    m0 = make_model_for_name(first_name, device)
    state = torch.load(saved_checkpoints[first_name], map_location=device)
    m0.load_state_dict(state)
    m0.eval()
    _, val_loader_v = get_pytorch_dataloaders(
        X_train_full,
        y_train_full,
        X_val,
        y_val,
        batch_size=BATCH_SIZE,
        img_size=IMG_SIZE_CNN,
        normalize_train=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )
    wrong_idx = []
    start = 0
    with torch.no_grad():
        for images, labels in val_loader_v:
            images = images.to(device, non_blocking=True)
            if USE_AMP and device.type == "cuda":
                with torch.amp.autocast("cuda"):
                    pred = m0(images).argmax(dim=1).cpu().numpy()
            else:
                pred = m0(images).argmax(dim=1).cpu().numpy()
            labels_np = labels.numpy()
            for i in range(len(labels_np)):
                if pred[i] != labels_np[i]:
                    wrong_idx.append(start + i)
            start += len(labels_np)
    print(f"{first_name}: {len(wrong_idx)} val errors (first 20 indices): {wrong_idx[:20]}")


### Ensemble on val — mean P(dog), threshold search


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

val_norm = Tv2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)


def numpy_to_nchw(X_np: np.ndarray) -> torch.Tensor:
    t = torch.from_numpy(X_np).permute(0, 3, 1, 2).float()
    if (t.shape[2], t.shape[3]) != IMG_SIZE_CNN:
        t = torch.nn.functional.interpolate(
            t, size=IMG_SIZE_CNN, mode="bilinear", align_corners=False
        )
    return t


@torch.no_grad()
def dog_probs_tta(model, X_np: np.ndarray, desc: str = "TTA") -> np.ndarray:
    # Mean P(dog) over identity + horizontal flip (batched pairs).
    model.eval()
    t = numpy_to_nchw(X_np)
    t0 = val_norm(t)
    t1 = val_norm(torch.flip(t, dims=[3]))
    n = t0.shape[0]
    out = np.zeros(n, dtype=np.float32)
    for i in tqdm(range(0, n, BATCH_SIZE), desc=desc, leave=False):
        c0 = t0[i : i + BATCH_SIZE]
        c1 = t1[i : i + BATCH_SIZE]
        cat = torch.cat([c0, c1], dim=0).to(device, non_blocking=True)
        if USE_AMP and device.type == "cuda":
            with torch.amp.autocast("cuda"):
                logits = model(cat)
        else:
            logits = model(cat)
        prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        b = c0.shape[0]
        out[i : i + b] = 0.5 * (prob[:b] + prob[b:])
    return out


val_probs_by_model = {}
for arch_name, ckpt in saved_checkpoints.items():
    m = make_model_for_name(arch_name, device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m = m.to(device)
    m.eval()
    val_probs_by_model[arch_name] = dog_probs_tta(m, X_val, desc=f"Val TTA {arch_name}")

stack = np.stack([val_probs_by_model[k] for k in val_probs_by_model], axis=0)
mean_dog_val = stack.mean(axis=0)

best_t = 0.5
best_acc = -1.0
for t in THRESHOLD_GRID:
    pred = (mean_dog_val >= t).astype(np.int64)
    acc = (pred == y_val).mean()
    if acc > best_acc:
        best_acc = acc
        best_t = float(t)

print(f"Val: best threshold on mean P(dog): {best_t:.3f}  acc={best_acc:.4f}")


### Test prediction — ensemble TTA + submission CSV


In [ ]:
X_test, test_ids = load_test_images(img_size=IMG_SIZE_CNN, grayscale=False)
print(f"Test: {X_test.shape}, ids: {len(test_ids)}")

test_probs_by_model = {}
for arch_name, ckpt in saved_checkpoints.items():
    m = make_model_for_name(arch_name, device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m = m.to(device)
    test_probs_by_model[arch_name] = dog_probs_tta(m, X_test, desc=f"Test TTA {arch_name}")

stack_te = np.stack([test_probs_by_model[k] for k in test_probs_by_model], axis=0)
mean_dog_te = stack_te.mean(axis=0)
pred_labels = (mean_dog_te >= best_t).astype(np.int64)

submission_path = generate_submission_csv(
    test_ids, pred_labels, SUBMISSIONS_DIR / "submission_optionC_ensemble.csv"
)
print(f"Saved: {submission_path} ({len(test_ids)} rows)")
print(f"  Models: {list(saved_checkpoints.keys())}")
print(f"  TTA: identity + hflip, ensemble mean, threshold={best_t:.3f}")
print(f"  Dogs: {(pred_labels == 1).sum()}, Cats: {(pred_labels == 0).sum()}")
